## Mounting Drive and Extracting the Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

extract_dir = '/content/extracted_sahi'
os.makedirs(extract_dir, exist_ok=True)

!tar -xf /content/drive/MyDrive/Yolov7/sahi_dataset.tar -C {extract_dir}

print(f"Extraction complete! Files are located in {extract_dir}")

Extraction complete! Files are located in /content/extracted_sahi


## Downloading and Training the Model

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 37.0 MB/s eta 0:00:00


In [ ]:
path = "/content/extracted_sahi/sahi/SeaDronesSee_SAHI.yaml"

with open(path, "r") as f:
    print(f.read())

# SeaDronesSee — SAHI pre-processed dataset config for YOLOv7
# Generated automatically by prepare_sahi_dataset.py

train: /content/project/yolov7/data/sahi/train/images
val:   /content/project/yolov7/data/sahi/val/images
test:  /content/project/yolov7/data/test/images

nc: 7
names: ['ignored', 'swimmer', 'boat', 'jetski', 'life_saving_appliances', 'buoy']



In [ ]:
import yaml

with open("/content/extracted_sahi/sahi/SeaDronesSee_SAHI.yaml") as f:
    data = yaml.safe_load(f)

print("nc =", data["nc"])
print("len(names) =", len(data["names"]))
print(data["names"])

nc = 7
len(names) = 6
['ignored', 'swimmer', 'boat', 'jetski', 'life_saving_appliances', 'buoy']


In [ ]:
import yaml

path = "/content/extracted_sahi/sahi/SeaDronesSee_SAHI.yaml"

with open(path, "r") as f:
    data = yaml.safe_load(f)

data["nc"] = len(data["names"])

with open(path, "w") as f:
    yaml.dump(data, f)

print("YAML fixed successfully!")

YAML fixed successfully!


In [ ]:
with open("/content/extracted_sahi/sahi/SeaDronesSee_SAHI.yaml") as f:
    data = yaml.safe_load(f)

print("nc =", data["nc"])
print("names =", len(data["names"]))

nc = 6
names = 6


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov10n.pt')

results = model.train(
    data='/content/extracted_sahi/sahi/SeaDronesSee_SAHI.yaml',
    epochs=10,
    imgsz=640,
    batch=128,
    project='/content/project/',
    name='yolov10n_sahi'
)

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/extracted_sahi/sahi/SeaDronesSee_SAHI.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov10n_sahi-2, nbs=64, nms=False, opset=

In [ ]:
from ultralytics import YOLO

best_model_path = '/content/project/yolov10n_sahi-2/weights/best.pt'
model = YOLO(best_model_path)

print("Evaluating on test dataset...")
metrics = model.val(data='/content/extracted_sahi/sahi/SeaDronesSee_SAHI.yaml', split='test')

Evaluating on test dataset...
Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
YOLOv10n summary (fused): 102 layers, 2,266,338 parameters, 0 gradients, 6.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 7395.9±3051.4 MB/s, size: 225.3 KB)
val: Scanning /content/extracted_sahi/sahi/test/labels... 7402 images, 3124 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 10526/10526 6.5Kit/s 1.6s
val: /content/extracted_sahi/sahi/test/images/13831_6132_1280_440_1920_1080.png: 1 duplicate labels removed
val: /content/extracted_sahi/sahi/test/images/9566_3677_2304_1152_2944_1792.png: 1 duplicate labels removed
val: /content/extracted_sahi/sahi/test/images/9570_3681_2304_1152_2944_1792.png: 1 duplicate labels removed
val: New cache created: /content/extracted_sahi/sahi/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 658/658 43.0it/s 15.3s
           

## Saving the model back to the drive

In [ ]:
!tar -cf /content/yolov10n.tar -C /content/ project

In [ ]:
!rsync -ah --progress /content/yolov10n.tar /content/drive/MyDrive/Yolov7/

print("Copy to Google Drive complete!")

sending incremental file list
yolov10n.tar
         20.03M 100%  794.60MB/s    0:00:00 (xfr#1, to-chk=0/1)
Copy to Google Drive complete!
